# Journal de bord

## 📅 Journal de bord – Jour 1 (01/09/2026)

### 🔧 Étapes / Actions réalisées
- Tirage du binôme et appropriation de la reprise M1 : modèle CISIA et API `model` fournis.
- Conception de l'architecture trois services `model` / `backend` / `frontend` dans `docker-compose.yml`.
- Ajout des healthchecks et des dépendances `model → backend → frontend`.
- Vérification du démarrage reproductible avec `docker compose config` et `docker compose up --build`.
- Mise en place du pipeline GitHub Actions : lint, tests, contract test, évaluation du modèle, build des trois images et publication GHCR.
- Ajout des tags Docker SHA, `latest` et tag de release `v*`, ainsi que de la création de GitHub Release.


### 📝 Observations / Difficultés
- Les premiers essais GitHub ont été réalisés manuellement avant de stabiliser le workflow CI/CD.
- La distinction entre tests unitaires, contract test, quality gate et publication GHCR a demandé une clarification.
- Le pipeline initial ne construisait que l'image `model`; il a été étendu aux images `model`, `backend` et `frontend`.
- Le modèle, le backend et le frontend utilisent des contrats distincts : il a fallu vérifier les ports, les healthchecks et les dépendances de démarrage.
- Les suites modèle et backend possèdent toutes deux un package local nommé `app`, ce qui a créé un conflit d'import lors de l'exécution globale de pytest.


### ✅ Prochaines étapes
- Instrumenter `/metrics` côté modèle et backend.
- Ajouter Prometheus et Grafana au compose.
- Construire le dashboard custom et le runbook d'astreinte.
- Vérifier le parcours complet de scoring avant de commencer la boucle de feedback.


---

## 📅 Journal de bord – Jour 2 (02/09/2026)

### 🔧 Étapes / Actions réalisées
- Endpoint `/metrics` exposé côté `model` et `backend`.
- Ajout des métriques HTTP : latence, RPS et codes de retour.
- Ajout de métriques métier : erreurs upstream, latence d'appel interne, répartition des classes et distribution des probabilités.
- Prometheus configuré pour scraper `model:8000/metrics` et `backend:8001/metrics`.
- Grafana ajouté au compose avec datasource Prometheus et dashboard provisionné automatiquement.
- Dashboard organisé selon les trois questions client :
  - vie : disponibilité, RPS et erreurs ;
  - vitesse : p50, p95 et p99 ;
  - comportement : classes prédites, confiance et classe dominante.
- `runbook.md` rédigé avec quatre procédures : service KO, latence, métrique modèle et rollback.
- README complété avec l'architecture, les ports et les commandes de démarrage.


### 📝 Observations / Difficultés
- Le provisionnement Grafana et les requêtes PromQL ont nécessité plusieurs vérifications de cohérence avec les noms réels des métriques.
- Les seuils p95 et les panels de comportement doivent être interprétés comme des signaux d'alerte, pas comme une preuve automatique de dérive.
- Le runbook a été relié aux panels du dashboard afin de transformer les métriques en actions opérationnelles.
- Un point important a été identifié pour la suite : un workflow GitHub Actions ne peut pas lire directement le volume SQLite local de production.


### ✅ Prochaines étapes
- Construire la boucle de feedback M5-B2 à partir du `request_id` de chaque prédiction.
- Ajouter le registre des prédictions et l'interface conseiller.
- Adapter le réentraînement au modèle CISIA multiclasses.
- Définir l'évaluation candidat/production et la politique de promotion.
- Tester le déclenchement, le déploiement et le rollback.


### ✅ Prochaines étapes
- Construire la boucle de feedback M5-B2 à partir du `request_id` de chaque prédiction.
- Ajouter le registre des prédictions et l'interface conseiller.
- Adapter le réentraînement au modèle CISIA multiclasses.
- Définir l'évaluation candidat/production et la politique de promotion.
- Tester le déclenchement, le déploiement et le rollback.

---

## 📅 Journal de bord – M5-B2 / boucle de rétroaction

### 🔧 Étapes / Actions réalisées
- Ajout d'un registre SQLite des prédictions dans le backend : entrée brute, prédiction, probabilités, version du modèle et date.
- Ajout de `POST /feedback` et `GET /feedback/count`.
- Validation du feedback : `404` pour un `request_id` inconnu, `409` pour un conflit, idempotence pour un rejeu identique et labels CISIA `0/1/2`.
- Ajout du volume Docker `feedback_data` pour conserver les annotations après redémarrage.
- Ajout dans le frontend d'un formulaire conseiller : le `request_id` retourné par `/score` est affiché et propagé au feedback.
- Adaptation du retrain à la pipeline multimodale CISIA avec preprocessing existant et `XGBClassifier` multiclasses.
- Mise en place du seuil de feedbacks non consommés : le retrain est ignoré sous le seuil configuré.
- Évaluation du candidat et de la production sur le même `data/reference_set.csv`.
- Politique de promotion fondée sur les planchers de qualité et la protection du recall de la classe 2.
- Journalisation des décisions dans `decisions_log.jsonl` et génération d'artefacts candidat/promu séparés.
- Ajout du service Compose `retrainer`, des scripts de lancement PowerShell/Linux et du workflow GitHub Actions planifié ou manuel.
- Rendre la promotion déployable avec `MODEL_ARTIFACT`, montage des artefacts modèles et scripts de déploiement/rollback.

### 🧪 Tests et validations
- `python -m pytest -q` : 22 tests passent.
- Tests backend et modèle exécutés séparément puis collectés globalement.
- Tests de feedback : stockage, `404`, `409`, idempotence et comptage des feedbacks non consommés.
- Tests de seuil : `199` feedbacks ne déclenchent pas le retrain ; `200` déclenchent le traitement.
- Test end-to-end isolé : feedback → retrain → évaluation → décision → consommation du feedback.
- Test du modèle dégradé avec `--degrade` : la quality gate retourne un code non nul.
- Tests de promotion : candidat accepté avec gain, candidat refusé sur régression de la classe 2.
- `docker compose config` et `docker compose --profile retrain config` passent.
- Construction de l'image `retrainer` et du frontend validée.

### 📝 Observations / Difficultés
- Le retrain CISIA est coûteux car il combine TF-IDF, preprocessing multimodal et 400 arbres XGBoost ; aucun log intermédiaire n'était affiché pendant `fit()`.
- `docker compose run --rm` supprime le conteneur à la fin : les logs ne sont plus disponibles après exécution, d'où l'importance de conserver les artefacts et le journal de décision dans un volume ou un artefact CI.
- Un runner GitHub Actions hébergé n'a pas accès au volume SQLite local ; le workflow distant doit recevoir une copie persistante de la base pour effectuer un vrai retrain.
- Le package local `app` existe dans les services model et backend ; les fixtures de tests ont été isolées pour éviter les conflits d'import.

### ✅ Bilan et prochaines étapes
- L'architecture M5 est testée de bout en bout : scoring, feedback, retrain, évaluation, promotion et déploiement.
- La boucle est opérationnelle localement avec Docker Compose.
- Prochaine amélioration : connecter la base de feedback à un stockage partagé ou distant pour rendre le retrain GitHub Actions pleinement automatique.
- Prochaine amélioration : ajouter des logs de progression et une durée maximale au retrain.
- Prochaine amélioration : intégrer le déploiement de l'artefact promu à la release CI/CD.
